# Home Energy-Usage Disaggregation (NILM) — Kaggle Training Notebook

**Architecture**: Single Shared Conv1D-BiLSTM Encoder + Multi-Appliance Dual-Branch Heads (Seq2Seq)

**Features**:
- ⚡ **Single joint model** disaggregates Fridge, Microwave, Dishwasher, Washing Machine simultaneously.
- ⏱️ **6-second uniform sampling**, sliding window $L = 599$ samples (~1 hour).
- 📊 **Active-period normalization** for targets + global normalization for aggregate mains.
- 🚀 **PyTorch Mixed Precision (AMP)** for ultra-fast GPU training.
- 💾 **Persistent Checkpointing** resilient to Kaggle kernel resets.
- 🏠 **Cross-Household Generalization**: Unseen test evaluation on held-out House 2.

In [ ]:
# Check GPU Availability and environment
import os, sys, torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Under Notebook Options -> Accelerator, select GPU.")
    is_interactive = not hasattr(sys, 'ps1') and ('KAGGLE_KERNEL_RUN_TYPE' in os.environ and os.environ['KAGGLE_KERNEL_RUN_TYPE'] == 'Interactive')
    if not is_interactive:
        print("Notebook verified and uploaded successfully. To run training, open in Kaggle editor and select GPU.")
        sys.exit(0)


## 1. Install & Imports

In [ ]:
import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Checkpoint directory (mount a Kaggle dataset or directory)
PERSISTENT_CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
PERSISTENT_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Configuration & Hyperparameters

In [ ]:
APPLIANCES = ["fridge", "microwave", "dishwasher", "washing_machine"]
THRESHOLDS = {
    "fridge": 50.0,
    "microwave": 200.0,
    "dishwasher": 10.0,
    "washing_machine": 20.0,
}
WINDOW_LEN = 599
TRAIN_STRIDE = 599 // 4  # 149 samples
VAL_STRIDE = 599
SAMPLE_PERIOD = 6  # seconds
BATCH_SIZE = 128
LR = 1e-3
LAMBDA_BCE = 1.0
EPOCHS = 35
PATIENCE = 6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Model Architecture (Shared Conv1D-BiLSTM + Multi-Head)

In [ ]:
class SharedEncoder(nn.Module):
    def __init__(self, in_channels=1, dropout=0.2, lstm_hidden=128):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 32, kernel_size=9, padding=4)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=7, padding=3)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden,
            num_layers=1, bidirectional=True, batch_first=True
        )

    def forward(self, x):
        # (B, L, 1) -> (B, 1, L)
        if x.dim() == 3 and x.shape[-1] == 1:
            x = x.permute(0, 2, 1)
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.dropout(h)
        h = h.permute(0, 2, 1)  # (B, L, C)
        out, _ = self.lstm(h)   # (B, L, 256)
        return out

class ApplianceHead(nn.Module):
    def __init__(self, in_features=256, conv_filters=64, dense_dim=32):
        super().__init__()
        self.conv = nn.Conv1d(in_features, conv_filters, kernel_size=3, padding=1)
        self.dense = nn.Linear(conv_filters, dense_dim)
        self.power_regressor = nn.Linear(dense_dim, 1)
        self.onoff_classifier = nn.Linear(dense_dim, 1)

    def forward(self, encoder_features):
        h = encoder_features.permute(0, 2, 1)
        h = F.relu(self.conv(h))
        h = h.permute(0, 2, 1)
        h = F.relu(self.dense(h))
        power = self.power_regressor(h)
        onoff = torch.sigmoid(self.onoff_classifier(h))
        return power, onoff

class MultiApplianceNILM(nn.Module):
    def __init__(self, appliances):
        super().__init__()
        self.appliances = appliances
        self.encoder = SharedEncoder()
        self.heads = nn.ModuleDict({app: ApplianceHead() for app in appliances})

    def forward(self, x):
        features = self.encoder(x)
        p_list, o_list = [], []
        for app in self.appliances:
            p, o = self.heads[app](features)
            p_list.append(p)
            o_list.append(o)
        return {
            "power": torch.cat(p_list, dim=-1),
            "on_off": torch.cat(o_list, dim=-1)
        }

## 4. Multi-Task Loss Function

In [ ]:
class MultiApplianceLoss(nn.Module):
    def __init__(self, appliances, lambda_bce=1.0):
        super().__init__()
        self.appliances = appliances
        self.lambda_bce = lambda_bce

    def forward(self, power_pred, power_true, onoff_pred, onoff_true):
        eps = 1e-7
        onoff_pred = torch.clamp(onoff_pred, eps, 1.0 - eps)
        total_loss = 0.0
        for i, _ in enumerate(self.appliances):
            mse = F.mse_loss(power_pred[..., i], power_true[..., i])
            bce = F.binary_cross_entropy(onoff_pred[..., i], onoff_true[..., i])
            total_loss = total_loss + (mse + self.lambda_bce * bce)
        return total_loss

## 5. Dataset Loading & Sliding Window Generator

In [ ]:
# If using local repository, import directly from src:
import sys
sys.path.append(".")
try:
    from src.train import prepare_datasets, train_model
    from src.evaluate import run_evaluation
    from src.config import NILMConfig
    print("Successfully imported NILM modules from src.")
except ImportError:
    print("Running self-contained mode in Kaggle.")

## 6. Train Model with Mixed Precision (AMP)

In [ ]:
config = NILMConfig(
    epochs=35,
    batch_size=128,
    lr=1e-3,
    checkpoint_dir=str(PERSISTENT_CHECKPOINT_DIR),
    device=DEVICE
)

train_ds, val_ds, test_ds, norm_params = prepare_datasets(config)
model, history = train_model(config, train_ds, val_ds, norm_params, checkpoint_dir=str(PERSISTENT_CHECKPOINT_DIR))

## 7. Evaluate Cross-Household Generalization

In [ ]:
results, comp_df = run_evaluation(
    checkpoint_path=str(PERSISTENT_CHECKPOINT_DIR / "best_model.pt"),
    device=DEVICE
)
comp_df